# 02 — Silver Layer Cleaning
**Project**  : Olist Brazilian E-Commerce Data Warehouse  
**Layer**    : Silver (Cleaned & Validated)  
**Author**   : Salman  
**Platform** : Databricks (Unity Catalog + Delta Lake)  

---

## Purpose
This notebook reads from the Bronze layer, applies data quality fixes, and writes cleaned data  
into the Silver layer as Delta Tables using **Spark SQL**.

Silver is the **cleaned version** of Bronze — no business aggregations, just standardization and known issue handling.

## Transformations Applied
| Table | Issue Found | Fix Applied |
|-------|------------|-------------|
| geolocation | Corrupt encoding in city names, coordinate outliers outside Brazil, multiple city names per zip code | Filter corrupt rows, filter by Brazil bounding box, keep most frequent city per zip code using median coordinates |
| order_reviews | 789 duplicate `review_id` assigned to multiple `order_id` | Deduplicate using `ROW_NUMBER()`, keep one row per `review_id` |
| order_payments | 2 rows with `payment_installments = 0` on `credit_card` | Replace 0 with 1 |
| products | 610 rows with NULL `product_category_name` | Replace with `'unknown'` |
| sellers | 3 rows with corrupt encoding in `seller_city` | Exclude those rows |
| customers, order_items, orders, product_category_name_translation | No issues found | Pass-through |

## Source → Target
- **Source** : `workspace.bronze.*`
- **Target** : `workspace.silver.*`

## 0. Setup & Configuration

In [0]:
from datetime import datetime

def log(msg):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {msg}")

log("Setup complete.")
log("Source : workspace.bronze")
log("Target : workspace.silver")

## 1. Create Silver Schema (if not exists)

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.silver")
log("Schema workspace.silver is ready.")

## 2. Silver — customers

**Issues found** : None  
**Action** : Pass-through — copy as-is from bronze, add `dwh_create_date`

In [0]:
start = datetime.now()
log(">> Processing: customers")

spark.sql("""
    CREATE OR REPLACE TABLE workspace.silver.customers
    USING DELTA AS
    SELECT
        customer_id,
        customer_unique_id,
        customer_zip_code_prefix,
        customer_city,
        customer_state,
        current_timestamp() AS dwh_create_date
    FROM workspace.bronze.customers
""")

count = spark.sql("SELECT COUNT(*) AS cnt FROM workspace.silver.customers").collect()[0]["cnt"]
elapsed = (datetime.now() - start).seconds
log(f">> Done: silver.customers — {count:,} rows — {elapsed}s")

## 3. Silver — geolocation

**Issues found**:
- Corrupt UTF-8 encoding in `geolocation_city` (e.g. `s├úo paulo` instead of `são paulo`)
- Coordinate outliers outside Brazil bounding box (`lat` -33.75 to 5.27, `lng` -73.99 to -34.79)
- Multiple city names per zip code prefix due to typos and inconsistent entries

**Fix applied**:
1. Filter out rows with coordinates outside Brazil bounding box
2. Normalize city (LOWER + TRIM) and state (UPPER + TRIM)
3. Compute median `lat`/`lng` per zip code prefix (more robust than AVG against outliers)
4. Keep the most frequent city name per zip code prefix (mode)
5. Result: **one row per zip code prefix** in silver

In [0]:
start = datetime.now()
log(">> Processing: geolocation")

spark.sql("""
    CREATE OR REPLACE TABLE workspace.silver.geolocation
    USING DELTA AS
    WITH cleaned AS (
        SELECT
            geolocation_zip_code_prefix,
            geolocation_lat,
            geolocation_lng,
            LOWER(TRIM(geolocation_city))  AS geolocation_city,
            UPPER(TRIM(geolocation_state)) AS geolocation_state
        FROM workspace.bronze.geolocation
        WHERE
            -- Filter coordinates outside Brazil bounding box
            geolocation_lat BETWEEN -33.75 AND 5.27
            AND geolocation_lng BETWEEN -73.99 AND -34.79
    ),
    grouped AS (
        SELECT
            geolocation_zip_code_prefix,
            geolocation_city,
            geolocation_state,
            -- Median is more robust than AVG against coordinate outliers
            PERCENTILE_APPROX(geolocation_lat, 0.5)
                OVER (PARTITION BY geolocation_zip_code_prefix) AS geolocation_lat,
            PERCENTILE_APPROX(geolocation_lng, 0.5)
                OVER (PARTITION BY geolocation_zip_code_prefix) AS geolocation_lng,
            COUNT(*)
                OVER (PARTITION BY geolocation_zip_code_prefix, geolocation_city, geolocation_state) AS freq
        FROM cleaned
    ),
    ranked AS (
        SELECT DISTINCT
            geolocation_zip_code_prefix,
            geolocation_city,
            geolocation_state,
            geolocation_lat,
            geolocation_lng,
            ROW_NUMBER()
                OVER (PARTITION BY geolocation_zip_code_prefix ORDER BY freq DESC) AS rn
        FROM grouped
    )
    SELECT
        geolocation_zip_code_prefix,
        geolocation_lat,
        geolocation_lng,
        geolocation_city,
        geolocation_state,
        current_timestamp() AS dwh_create_date
    FROM ranked
    WHERE rn = 1
""")

count = spark.sql("SELECT COUNT(*) AS cnt FROM workspace.silver.geolocation").collect()[0]["cnt"]
elapsed = (datetime.now() - start).seconds
log(f">> Done: silver.geolocation — {count:,} rows (1 row per zip code prefix) — {elapsed}s")

## 4. Silver — order_items

**Issues found** : None  
**Action** : Pass-through — copy as-is from bronze, add `dwh_create_date`

In [0]:
start = datetime.now()
log(">> Processing: order_items")

spark.sql("""
    CREATE OR REPLACE TABLE workspace.silver.order_items
    USING DELTA AS
    SELECT
        order_id,
        order_item_id,
        product_id,
        seller_id,
        shipping_limit_date,
        price,
        freight_value,
        current_timestamp() AS dwh_create_date
    FROM workspace.bronze.order_items
""")

count = spark.sql("SELECT COUNT(*) AS cnt FROM workspace.silver.order_items").collect()[0]["cnt"]
elapsed = (datetime.now() - start).seconds
log(f">> Done: silver.order_items — {count:,} rows — {elapsed}s")

## 5. Silver — order_payments

**Issues found**:
- 2 rows with `payment_installments = 0` where `payment_type = 'credit_card'`  
  (credit card must have at least 1 installment — data entry error)

**Fix applied**: Replace `payment_installments = 0` with `1` for `credit_card` rows only

In [0]:
start = datetime.now()
log(">> Processing: order_payments")

spark.sql("""
    CREATE OR REPLACE TABLE workspace.silver.order_payments
    USING DELTA AS
    SELECT
        order_id,
        payment_sequential,
        payment_type,
        CASE
            WHEN payment_installments = 0
             AND payment_type = 'credit_card' THEN 1
            ELSE payment_installments
        END AS payment_installments,
        payment_value,
        current_timestamp() AS dwh_create_date
    FROM workspace.bronze.order_payments
""")

count = spark.sql("SELECT COUNT(*) AS cnt FROM workspace.silver.order_payments").collect()[0]["cnt"]
elapsed = (datetime.now() - start).seconds
log(f">> Done: silver.order_payments — {count:,} rows — {elapsed}s")

## 6. Silver — order_reviews

**Issues found**:
- 789 duplicate `review_id` assigned to more than one `order_id`  
  (764 appear in 2 orders, 25 appear in 3 orders — all other columns are identical, source system bug)

**Fix applied**: Deduplicate using `ROW_NUMBER()` partitioned by `review_id`, ordered by `order_id` (deterministic).  
**Known limitation**: Selected `order_id` for duplicated reviews may not be the original one. Impact is small (789 / 99,224 rows).

In [0]:
start = datetime.now()
log(">> Processing: order_reviews")

spark.sql("""
    CREATE OR REPLACE TABLE workspace.silver.order_reviews
    USING DELTA AS
    WITH ranked AS (
        SELECT
            review_id,
            order_id,
            review_score,
            review_comment_title,
            review_comment_message,
            review_creation_date,
            review_answer_timestamp,
            ROW_NUMBER() OVER (
                PARTITION BY review_id
                ORDER BY order_id
            ) AS rn
        FROM workspace.bronze.order_reviews
    )
    SELECT
        review_id,
        order_id,
        review_score,
        review_comment_title,
        review_comment_message,
        review_creation_date,
        review_answer_timestamp,
        current_timestamp() AS dwh_create_date
    FROM ranked
    WHERE rn = 1
""")

count = spark.sql("SELECT COUNT(*) AS cnt FROM workspace.silver.order_reviews").collect()[0]["cnt"]
elapsed = (datetime.now() - start).seconds
log(f">> Done: silver.order_reviews — {count:,} rows — {elapsed}s")

## 7. Silver — orders

**Issues found**:
- 8 orders with `order_status = 'delivered'` but `order_delivered_customer_date` is NULL (source system bug, cannot be fixed)
- NULL values in date columns are **expected** depending on order lifecycle status

**Action** : Pass-through — copy as-is, NULLs retained as valid business values

In [0]:
start = datetime.now()
log(">> Processing: orders")

spark.sql("""
    CREATE OR REPLACE TABLE workspace.silver.orders
    USING DELTA AS
    SELECT
        order_id,
        customer_id,
        order_status,
        order_purchase_timestamp,
        order_approved_at,
        order_delivered_carrier_date,
        order_delivered_customer_date,
        order_estimated_delivery_date,
        current_timestamp() AS dwh_create_date
    FROM workspace.bronze.orders
""")

count = spark.sql("SELECT COUNT(*) AS cnt FROM workspace.silver.orders").collect()[0]["cnt"]
elapsed = (datetime.now() - start).seconds
log(f">> Done: silver.orders — {count:,} rows — {elapsed}s")

## 8. Silver — products

**Issues found**:
- 610 rows with NULL `product_category_name` — replaced with `'unknown'`
- 2 rows with NULL dimensions/weight — kept as-is (referenced by 17 transactions)
- 4 rows with `product_weight_g = 0` — kept as-is (referenced by 8 transactions)

**Fix applied**: Replace NULL `product_category_name` with `'unknown'`

In [0]:
start = datetime.now()
log(">> Processing: products")

spark.sql("""
    CREATE OR REPLACE TABLE workspace.silver.products
    USING DELTA AS
    SELECT
        product_id,
        COALESCE(product_category_name, 'unknown') AS product_category_name,
        product_name_lenght,
        product_description_lenght,
        product_photos_qty,
        product_weight_g,
        product_length_cm,
        product_height_cm,
        product_width_cm,
        current_timestamp() AS dwh_create_date
    FROM workspace.bronze.products
""")

count = spark.sql("SELECT COUNT(*) AS cnt FROM workspace.silver.products").collect()[0]["cnt"]
elapsed = (datetime.now() - start).seconds
log(f">> Done: silver.products — {count:,} rows — {elapsed}s")

## 9. Silver — sellers

**Issues found**:
- 3 rows with corrupt UTF-8 encoding in `seller_city` (e.g. `santa barbara d┬┤oeste`)
- Many inconsistent city name formats — kept as-is per medallion architecture best practice;  
  city enrichment will be done in gold layer via join to `silver.geolocation`

**Fix applied**:
- Exclude 3 rows with corrupt encoding (detected via box-drawing character pattern)
- Normalize: `seller_city` → `LOWER + TRIM`, `seller_state` → `UPPER + TRIM`

In [0]:
start = datetime.now()
log(">> Processing: sellers")

spark.sql("""
    CREATE OR REPLACE TABLE workspace.silver.sellers
    USING DELTA AS
    SELECT
        seller_id,
        seller_zip_code_prefix,
        LOWER(TRIM(seller_city))  AS seller_city,
        UPPER(TRIM(seller_state)) AS seller_state,
        current_timestamp()       AS dwh_create_date
    FROM workspace.bronze.sellers
    WHERE
        -- Exclude rows with corrupt UTF-8 encoding in seller_city
        -- Identified by presence of box-drawing characters (┬, └, ├)
        seller_city NOT RLIKE '[\\u251c\\u2514\\u252c]'
""")

count = spark.sql("SELECT COUNT(*) AS cnt FROM workspace.silver.sellers").collect()[0]["cnt"]
elapsed = (datetime.now() - start).seconds
log(f">> Done: silver.sellers — {count:,} rows — {elapsed}s")

## 10. Silver — product_category_name_translation

**Issues found** : None  
**Action** : Pass-through — copy as-is from bronze, add `dwh_create_date`

In [0]:
start = datetime.now()
log(">> Processing: product_category_name_translation")

spark.sql("""
    CREATE OR REPLACE TABLE workspace.silver.product_category_name_translation
    USING DELTA AS
    SELECT
        product_category_name,
        product_category_name_english,
        current_timestamp() AS dwh_create_date
    FROM workspace.bronze.product_category_name_translation
""")

count = spark.sql("SELECT COUNT(*) AS cnt FROM workspace.silver.product_category_name_translation").collect()[0]["cnt"]
elapsed = (datetime.now() - start).seconds
log(f">> Done: silver.product_category_name_translation — {count:,} rows — {elapsed}s")

## 11. Final Validation — Row Count Check

In [0]:
from datetime import datetime

log("=" * 65)
log("FINAL VALIDATION — BRONZE vs SILVER ROW COUNTS")
log("=" * 65)

tables = [
    "customers",
    "geolocation",
    "order_items",
    "order_payments",
    "order_reviews",
    "orders",
    "products",
    "sellers",
    "product_category_name_translation",
]

print(f"\n{'Table':<45} {'Bronze':>12} {'Silver':>12} {'Diff':>8}")
print("-" * 80)

for tbl in tables:
    bronze = spark.sql(f"SELECT COUNT(*) AS cnt FROM workspace.bronze.{tbl}").collect()[0]["cnt"]
    silver = spark.sql(f"SELECT COUNT(*) AS cnt FROM workspace.silver.{tbl}").collect()[0]["cnt"]
    diff   = silver - bronze
    diff_str = str(diff) if diff == 0 else f"{diff:+,}"
    print(f"{tbl:<45} {bronze:>12,} {silver:>12,} {diff_str:>8}")

print("-" * 80)
log("Validation complete. Review differences above.")
log("Expected differences:")
log("  geolocation   : large reduction (deduplicated to 1 row per zip code prefix)")
log("  order_reviews : small reduction (~789 duplicate review_id removed)")
log("  sellers       : -3 rows (corrupt encoding excluded)")